## Cell 0 — Run ONCE, then Runtime → Restart Session

In [ ]:
# Cell 0 — Run ONCE, then Runtime → Restart session
import subprocess

# Bake protobuf fix into Python startup before TensorFlow loads
hack = 'import os\nos.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"\n'
with open("/usr/local/lib/python3.12/sitecustomize.py", "w") as f:
    f.write(hack)

subprocess.run(["pip", "install", "-q", "numpy==1.26.4", "--force-reinstall"], check=False)
subprocess.run(["pip", "install", "-q", "protobuf==4.25.9", "--force-reinstall"], check=False)
subprocess.run(["pip", "install", "-q",
    "transformers", "datasets", "evaluate",
    "accelerate", "sentencepiece", "sacrebleu", "unbabel-comet"], check=False)

print("✅ Done — now go to Runtime → Restart session")


✅ Done — now go to Runtime → Restart session


## Cell 1 — Memory Config (Run First After Every Restart)

In [ ]:
# Cell 1 — Run first after every restart
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Reinstall packages — they don't survive Colab restarts
import subprocess
subprocess.run(["pip", "install", "-q",
    "transformers", "datasets", "evaluate",
    "accelerate", "sentencepiece", "sacrebleu", "unbabel-comet"], check=False)

import gc
import torch

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def memory_status():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved() / 1e9
        total     = torch.cuda.get_device_properties(0).total_memory / 1e9
        free      = total - reserved
        print(f"GPU Memory → Used: {allocated:.2f}GB | Reserved: {reserved:.2f}GB | Free: {free:.2f}GB | Total: {total:.2f}GB")
    else:
        print("No GPU detected!")

print("✅ GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
memory_status()


✅ GPU: Tesla T4
GPU Memory → Used: 0.00GB | Reserved: 0.00GB | Free: 15.64GB | Total: 15.64GB


In [ ]:
!pip install -q "numpy==1.26.4" --force-reinstall
print("✅ Now go to Runtime → Restart session")

ERROR: Operation cancelled by user
✅ Now go to Runtime → Restart session


In [ ]:
!pip install evaluate

  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.


## Cell 2 — Mount Drive & Imports

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    NllbTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    TrainerCallback
)
import evaluate

MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG   = "eng_Latn"
TGT_LANG   = "hau_Latn"
MAX_LEN    = 128
OUTPUT_DIR = "/content/drive/MyDrive/nllb-hausa-waec"

print("✅ Imports done")
print(f"✅ Checkpoints will save to: {OUTPUT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Imports done
✅ Checkpoints will save to: /content/drive/MyDrive/nllb-hausa-waec


## Cell 3 — Load Validated Pairs from XLSX

In [ ]:
# Load ONLY validated pairs — Sheet1 is reserved for TTS/WER evaluation
df = pd.read_excel("/content/hausa_pq_speech.xlsx", sheet_name="validated")

pairs = []
skipped = 0

for _, row in df.iterrows():
    # --- Question pairs ---
    eng_q = str(row.get("question_text_english") or "").strip()
    hau_q = str(row.get("validated_question_text_hausa") or "").strip()

    if len(eng_q) >= 10 and len(hau_q) >= 10 and hau_q != "nan":
        pairs.append({"src": eng_q, "tgt": hau_q})
    else:
        skipped += 1

    # --- Answer pairs ---
    eng_a = str(row.get("correct_answer_english") or "").strip()
    hau_a = str(row.get("correct_answer_hausa") or "").strip()

    if len(eng_a) > 2 and len(hau_a) > 2 and hau_a != "nan":
        pairs.append({"src": eng_a, "tgt": hau_a})

del df
clear_memory()

print(f"✅ Total pairs: {len(pairs)} | Skipped (questions only): {skipped}")
print(f"   Sample Q   EN : {pairs[0]['src']}")
print(f"   Sample Q   HA : {pairs[0]['tgt']}")
print(f"   Sample ANS EN : {pairs[1]['src']}")
print(f"   Sample ANS HA : {pairs[1]['tgt']}")


✅ Total pairs: 2240 | Skipped (questions only): 464
   Sample Q   EN : The number of teats in a cow is A. two B. four C. six D. eight
   Sample Q   HA : Yawan mama nawane a jikin saniya Option A. biyu Option B. huɗu Option C. shida Option D. takwas
   Sample ANS EN : B. huɗu
   Sample ANS HA : Option B. huɗu


## Cell 4 — Train / Val / Test Split (80/20)

In [ ]:
# 80% train, 20% held-out test — test set NEVER seen during training
train_data, test_data = train_test_split(pairs, test_size=0.2, random_state=42)

# Further split train into 90/10 for train/val (used only for eval callbacks)
train_data, val_data = train_test_split(train_data, test_size=0.1, random_state=42)

del pairs
clear_memory()

train_ds    = Dataset.from_list(train_data)
val_ds      = Dataset.from_list(val_data)
test_ds_raw = Dataset.from_list(test_data)  # never tokenized for training

del train_data, val_data, test_data
clear_memory()

print(f"✅ Train: {len(train_ds)} | Val: {len(val_ds)} | Test (held-out): {len(test_ds_raw)}")


✅ Train: 1612 | Val: 180 | Test (held-out): 448


## Cell 5 — Tokenizer & Tokenize

In [ ]:
tokenizer = NllbTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG)

def preprocess(batch):
    model_inputs = tokenizer(
        batch["src"],
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        text_target=batch["tgt"],
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])
val_tokenized   = val_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])

del train_ds, val_ds
clear_memory()

print("✅ Tokenization done")
memory_status()


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1612 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

✅ Tokenization done
GPU Memory → Used: 0.00GB | Reserved: 0.00GB | Free: 15.64GB | Total: 15.64GB


## Cell 6 — Load Model

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f"✅ Model loaded — {model.num_parameters():,} parameters")
memory_status()


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Model loaded — 1,402,138,624 parameters
GPU Memory → Used: 0.00GB | Reserved: 0.00GB | Free: 15.64GB | Total: 15.64GB


## Cell 7 — Metrics

In [ ]:
chrf = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)
    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = chrf.compute(predictions=decoded_preds, references=decoded_labels)
    del decoded_preds, decoded_labels
    clear_memory()
    return {"chrf": round(result["score"], 2)}

print("✅ Metrics ready")


✅ Metrics ready


In [ ]:
!pip install sacrebleu

  Using cached sacrebleu-2.6.0-py3-none-any.whl.metadata (39 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 

## Cell 8 — Training Arguments

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,      # effective batch = 32

    fp16=True,
    bf16=False,

    optim="adafactor",
    learning_rate=5e-5,
    warmup_steps=200,
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    gradient_checkpointing=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,

    predict_with_generate=True,
    generation_max_length=128,

    logging_steps=100,
    logging_first_step=True,
    report_to="none",
)
print("✅ Training args set")


✅ Training args set


## Cell 9 — Train (Auto-Resumes if Connection Drops)

In [ ]:
import os

# 1. Progress log callback — writes to Drive every 100 steps
class ProgressLogCallback(TrainerCallback):
    def __init__(self, log_path):
        self.log_path = log_path
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            with open(self.log_path, "a") as f:
                f.write(f"Step {state.global_step}: {logs}\n")

log_path = f"{OUTPUT_DIR}/training_log.txt"

# 2. Build Trainer
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2),
        ProgressLogCallback(log_path)
    ]
)

# 3. Auto-detect last checkpoint on Drive
last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [
        os.path.join(OUTPUT_DIR, d) for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"🔄 Resuming from checkpoint: {last_checkpoint}")
    else:
        print("🆕 No checkpoint found — starting fresh")

# 4. Train
print("Starting training...")
memory_status()
trainer.train(resume_from_checkpoint=last_checkpoint)


🆕 No checkpoint found — starting fresh
Starting training...
GPU Memory → Used: 5.62GB | Reserved: 5.62GB | Free: 10.01GB | Total: 15.64GB


Step,Training Loss,Validation Loss,Chrf
200,10.638645,1.241160,59.990000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=255, training_loss=16.640544966155407, metrics={'train_runtime': 2856.2844, 'train_samples_per_second': 2.822, 'train_steps_per_second': 0.089, 'total_flos': 3807355978383360.0, 'train_loss': 16.640544966155407, 'epoch': 5.0})

In [ ]:
# Run this, then re-run Cell 9
import shutil
import os

bad_checkpoint = "/content/drive/MyDrive/nllb-hausa-waec/checkpoint-500"

if os.path.exists(bad_checkpoint):
    shutil.rmtree(bad_checkpoint)
    print(f"🗑️ Deleted bad checkpoint: {bad_checkpoint}")

# Also check for any other incomplete checkpoints
checkpoint_dir = "/content/drive/MyDrive/nllb-hausa-waec"
if os.path.isdir(checkpoint_dir):
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
    print(f"Remaining checkpoints: {checkpoints if checkpoints else 'None — will start fresh'}")
p

🗑️ Deleted bad checkpoint: /content/drive/MyDrive/nllb-hausa-waec/checkpoint-500
Remaining checkpoints: None — will start fresh


-u
lo\\\\\
l









jn bnbnnm
## Cell 10 — Save Model

In [ ]:

                                                                              ave_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")

del trainer
clear_memory()

print(f"✅ Model saved to {OUTPUT_DIR}/final")
memory_status()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /content/drive/MyDrive/nllb-hausa-waec/final
GPU Memory → Used: 5.64GB | Reserved: 5.69GB | Free: 9.95GB | Total: 15.64GB


## Cell 11 — Quick Inference Check

In [ ]:
from transformers import pipeline

translator = pipeline(
    "translation",
    model=f"{OUTPUT_DIR}/final",
    tokenizer=tokenizer,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG,
    max_length=200,
    device=0
)

test_questions = [
    "Which of the following is not a function of the liver?",
    "Calculate the simple interest on 5000 naira at 8% per annum for 3 years.",
    "The process by which plants make their own food is called"
]

print("=== Translation Results ===")
for q in test_questions:
    result = translator(q)
    print(f"EN: {q}")
    print(f"HA: {result[0]['translation_text']}")
    print()


## Cell 12 — Generate Translations on Held-Out Test Set

> ⚠️ Run ONLY after training is complete. This 20% was never seen during training.

In [ ]:
import torch
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM

model_path     = f"{OUTPUT_DIR}/final"
tokenizer_eval = NllbTokenizer.from_pretrained(model_path, src_lang=SRC_LANG)
model_eval     = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda")
model_eval.eval()

BATCH_SIZE  = 16
tgt_lang_id = tokenizer_eval.convert_tokens_to_ids(TGT_LANG)

all_preds = []
all_refs  = []
src_texts = test_ds_raw["src"]
tgt_texts = test_ds_raw["tgt"]

print(f"Translating {len(src_texts)} test sentences...")

for i in range(0, len(src_texts), BATCH_SIZE):
    batch_src = src_texts[i : i + BATCH_SIZE]
    batch_tgt = tgt_texts[i : i + BATCH_SIZE]

    inputs = tokenizer_eval(
        batch_src,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    ).to("cuda")

    with torch.no_grad():
        generated = model_eval.generate(
            **inputs,
            forced_bos_token_id=tgt_lang_id,
            max_length=MAX_LEN,
            num_beams=4
        )

    decoded = tokenizer_eval.batch_decode(generated, skip_special_tokens=True)
    all_preds.extend(decoded)
    all_refs.extend(batch_tgt)

    if (i // BATCH_SIZE) % 10 == 0:
        print(f"  ... {i + len(batch_src)}/{len(src_texts)}")

clear_memory()
print("✅ Translations complete")


## Cell 13 — Compute BLEU / chrF / TER / AfriCOMET

In [ ]:
sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric      = evaluate.load("chrf")
ter_metric       = evaluate.load("ter")

refs_wrapped = [[r] for r in all_refs]

bleu_result = sacrebleu_metric.compute(predictions=all_preds, references=refs_wrapped)
chrf_result = chrf_metric.compute(predictions=all_preds, references=all_refs)
ter_result  = ter_metric.compute(predictions=all_preds, references=refs_wrapped)

print("=" * 50)
print("  HELD-OUT TEST SET RESULTS (20% split)")
print("=" * 50)
print(f"  BLEU      : {bleu_result['score']:.2f}")
print(f"  chrF      : {chrf_result['score']:.2f}")
print(f"  TER       : {ter_result['score']:.2f}")

try:
    from comet import download_model, load_from_checkpoint
    comet_model_path = download_model("masakhane/africomet-mtl")
    comet_model      = load_from_checkpoint(comet_model_path)
    comet_data = [
        {"src": s, "mt": p, "ref": r}
        for s, p, r in zip(src_texts, all_preds, all_refs)
    ]
    comet_output = comet_model.predict(comet_data, batch_size=16, gpus=1)
    print(f"  AfriCOMET : {comet_output.system_score:.4f}")
except Exception as e:
    print(f"  AfriCOMET : Could not compute ({e})")

print("=" * 50)


## Cell 14 — Spot-Check Sample Predictions

In [ ]:
import random
indices = random.sample(range(len(all_preds)), min(10, len(all_preds)))

print("=== Sample Test Predictions ===\n")
for idx in indices:
    print(f"SRC : {src_texts[idx]}")
    print(f"REF : {all_refs[idx]}")
    print(f"PRED: {all_preds[idx]}")
    print()
